# NB8: ESI/PCR Explainability Analysis

## Purpose

This notebook implements the two-dimensional explainability evaluation
framework:

**1. Explainability Stability Index (ESI)** -- measures SHAP attribution
stability under adversarial perturbation, grounded in Lipschitz continuity:

    ESI(x) = 1 - norm(phi(x_adv) - phi(x_clean)) / norm(x_adv - x_clean)

ESI=1: perfect stability. ESI=0: complete instability.

**2. Physics Constrained Ratio (PCR)** -- measures the fraction of total
SHAP attribution concentrated on physics-derived features:

    PCR = sum(|phi_i|, i in physics) / sum(|phi_j|, all j)

Physics features: voltage_imbalance, current_imbalance, power_factor,
freq_deviation.

**Five evaluation conditions:**
1. Standard IDS, clean data (baseline)
2. Standard IDS + PGD adversarial (Attack 1)
3. Robust IDS, clean data
4. Robust IDS + Adaptive PGD (Attack 3)

**Research questions answered:**
- Does PCR increase under attack? (preliminary finding: 0.35->0.45)
- Does adversarial training stabilise SHAP attributions (higher ESI)?
- Are physics-grounded classes (FDI) more explanation-stable under attack?
- Can SHAP-guided adversarial training restore explainability integrity?


---

**Fixed from the original Colab version:** loads local v4 artifacts
(`phase1_data_v4.pkl` / `_binary_v4.pkl`) instead of the stale Colab
`phase1_data.pkl`; loads XGBoost via `xgb.Booster()` instead of an unfitted
`XGBClassifier()`; per-class loops now use the real class count instead of a
hardcoded 6, so this also works on the binary track; supports both tracks via
the `BINARY` toggle in Setup. Regenerated for reproducibility/audit-trail --
`esi_pcr_results_v4.pkl` / `_binary_v4.pkl` already exist on disk from an
earlier run; this notebook now reproduces them from source.


---

**Further fixed (this revision) -- baseline-model cross-contamination.** This notebook loads `xgb_baseline_model{SUFFIX}.json` for the Standard IDS, correctly targeting the untuned baseline model. But two other artifacts were still pointing at the original tuned-pipeline's files: the PGD adversarial examples (`X_adv_pgd_eps01{SUFFIX}.npy`, crafted by attacking the *tuned* model's surrogate) and the defended model (`xgb_defended_model{SUFFIX}.json`, adversarially trained on top of the *tuned* model). This meant the "Robust IDS" condition here was silently identical to the tuned-model notebook's result (confirmed: matched to 4 decimal places across every per-class value) -- not a genuine baseline-pipeline robustness check. Fixed to load `X_adv_pgd_eps01_baseline{SUFFIX}.npy` (from the Attack 1 baseline notebook) and `xgb_defended_model_baseline{SUFFIX}.json` (from the Defence 1 baseline notebook). Output files renamed to `*_baseline{SUFFIX}` throughout so this run doesn't overwrite the original tuned-model results. Note this notebook's scope was always narrower than the other ESI/PCR notebook (4 conditions: Standard clean/PGD, Robust clean/Adaptive-PGD -- no PPO-unconstrained or PPO-budgeted conditions), so that gap is by original design, not something this fix introduces or needs to close.

---

**Further fixed (Drive-ready revision):** the 'Robust IDS + Adaptive PGD' condition was still loading `adaptive_surrogate_v4.pt` -- the *tuned* pipeline's adaptive surrogate. Now loads `adaptive_surrogate_baseline_v4.pt`, saved by the Attack 3 baseline notebook (NB6), so **NB8 must run after NB6**. Outputs were stripped; run fresh.


## Setup

In [ ]:
!pip install -q shap


In [ ]:
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import shap
import joblib
import xgboost as xgb
from sklearn.metrics import f1_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Drive-ready: PHYSAR-NEW lives on Google Drive (survives Colab runtime restarts)
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/PHYSAR-NEW'
os.chdir(PROJECT_DIR)
if not os.path.exists('phase1_data_v4.pkl'):
    print(f'WARNING: phase1_data_v4.pkl not found in {PROJECT_DIR} -- fix PROJECT_DIR before continuing.')
    print(sorted(os.listdir(PROJECT_DIR))[:20])
else:
    print('Found phase1_data_v4.pkl -- PROJECT_DIR looks correct.')

# ── Track toggle ─────────────────────────────────────────────────────────
# Run once with BINARY=False and once with BINARY=True to regenerate both
# esi_pcr_results_v4.pkl and esi_pcr_results_binary_v4.pkl (both already
# exist on disk from an earlier run elsewhere -- this notebook is being
# fixed/regenerated for reproducibility/audit-trail purposes).
BINARY = False
SUFFIX = '_binary_v4' if BINARY else '_v4'

RANDOM_STATE = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project dir: {os.getcwd()}')
print(f'Track: {"BINARY" if BINARY else "MULTICLASS"}')
print(f'Device: {DEVICE}')


## 1. Load all artifacts from Drive

In [ ]:
data = joblib.load(f'phase1_data{SUFFIX}.pkl')
X_train_scaled = data['X_train_scaled']
X_test_scaled  = data['X_test_scaled']
y_train = pd.Series(data['y_train'])
y_test  = pd.Series(data['y_test'])
FEATURE_COLS = data['FEATURE_COLS']
y_train_np = y_train.values
y_test_np  = y_test.values

N_CLASSES = int(len(np.unique(y_train_np)))
class_names = (['benign','backdoor','Bruteforce','FDI','ransomware','reverseshell']
               if N_CLASSES == 6 else [f'class_{i}' for i in range(N_CLASSES)])

class BoosterWrapper:
    """Wraps a raw xgb.Booster so it behaves like XGBClassifier for .predict().
    Same fix as NB9b(3)/(4)/NB_Ablation: XGBClassifier().load_model() on a
    never-fitted instance does not populate n_classes_, which crashes
    .predict() on multiclass models. Not strictly required for SHAP
    TreeExplainer (it reads booster internals directly) but used here for
    consistency, since no other artifact needs xgb_model.predict() elsewhere
    in this notebook -- if you add any, this keeps it safe."""
    def __init__(self, booster, n_classes):
        self.booster = booster
        self.n_classes_ = n_classes

    def predict(self, X):
        raw = self.booster.predict(xgb.DMatrix(X))
        if raw.ndim > 1:
            return raw.argmax(axis=1)
        return (raw > 0.5).astype(int)

# NOTE: 'xgb_tuned_model.json' from the old Colab notebook maps to
# xgb_baseline_model{SUFFIX}.json -- v4 never persisted a separately tuned
# model; the locked Standard IDS is the baseline (untuned) XGBoost.
_booster_std = xgb.Booster()
_booster_std.load_model(f'xgb_baseline_model{SUFFIX}.json')
xgb_model = BoosterWrapper(_booster_std, N_CLASSES)

_booster_def = xgb.Booster()
# FIXED: was loading xgb_defended_model{SUFFIX}.json -- the ORIGINAL tuned-pipeline's
# defended model, unrelated to the baseline Standard model just loaded above. This
# silently made every "Robust IDS" condition in this notebook identical to the
# tuned-model notebook's results (confirmed: values matched to 4 decimal places),
# not a genuine baseline-pipeline robustness result. Now loads the actual
# baseline-trained defended model from NB5's baseline conversion.
_booster_def.load_model(f'xgb_defended_model_baseline{SUFFIX}.json')
xgb_defended = BoosterWrapper(_booster_def, N_CLASSES)

# FIXED: was loading X_adv_pgd_eps01{SUFFIX}.npy -- PGD examples crafted by attacking
# the TUNED model's surrogate (from the original Attack 1 notebook), not the baseline
# model. Evaluating the baseline Standard model's SHAP stability against tuned-crafted
# perturbations is not a coherent baseline-pipeline test. Now loads the genuinely
# baseline-crafted PGD examples from NB3's baseline conversion.
X_adv_pgd = np.load(f'X_adv_pgd_eps01_baseline{SUFFIX}.npy')

PHYSICS_FEATURES = ['voltage_imbalance', 'current_imbalance',
                    'power_factor', 'freq_deviation']
PHYSICS_IDX = [FEATURE_COLS.index(f) for f in PHYSICS_FEATURES]
CYBER_FEATURES = [f for f in FEATURE_COLS if f.startswith('cyber_')]
CYBER_IDX = [FEATURE_COLS.index(f) for f in CYBER_FEATURES]

attack_mask = (y_test_np != 0)

print(f'Data loaded: {X_test_scaled.shape}  Classes: {N_CLASSES}')
print(f'Physics features ({len(PHYSICS_IDX)}): {PHYSICS_FEATURES}')
print(f'Cyber features ({len(CYBER_IDX)}): {len(CYBER_IDX)} features')


## 2. Stratified subsample + SHAP computation

In [ ]:
N_PER_CLASS_SHAP = 500
sample_idx = []
for cls_idx in range(N_CLASSES):
    cls_mask = np.where(y_test_np == cls_idx)[0]
    chosen = np.random.RandomState(RANDOM_STATE).choice(
        cls_mask, size=min(N_PER_CLASS_SHAP, len(cls_mask)), replace=False
    )
    sample_idx.extend(chosen.tolist())
sample_idx = np.array(sample_idx)

X_sample_clean = X_test_scaled[sample_idx]
X_sample_pgd   = X_adv_pgd[sample_idx]
y_sample       = y_test_np[sample_idx]

print(f'SHAP sample: {X_sample_clean.shape}')
print(f'Class distribution: {np.bincount(y_sample)}')

# Load adaptive surrogate and generate adaptive PGD.
# Reconstruct the architecture from the checkpoint's own weight shapes rather
# than assuming a fixed hidden size -- the saved adaptive_surrogate{SUFFIX}.pt
# turns out to be hidden=128 with an extra layer, not the hidden=64/3-layer
# shape a hardcoded class definition would assume (same class of bug already
# fixed for the regular surrogate_mlp in NB9b(3)/NB_Ablation).
_adap_state = torch.load(f'adaptive_surrogate_baseline{SUFFIX}.pt', map_location=DEVICE)
_adap_weight_keys = sorted([k for k in _adap_state if k.endswith('.weight')],
                           key=lambda x: int(x.split('.')[1]))
print('Adaptive surrogate checkpoint layers:',
      [(k, tuple(_adap_state[k].shape)) for k in _adap_weight_keys])

_adap_layers = []
for i, wk in enumerate(_adap_weight_keys):
    out_f, in_f = _adap_state[wk].shape
    _adap_layers.append(nn.Linear(in_f, out_f))
    if i < len(_adap_weight_keys) - 1:
        _adap_layers.append(nn.ReLU())
        _adap_layers.append(nn.Dropout(0.1))
adaptive_surrogate = nn.Sequential(*_adap_layers).to(DEVICE)

_adap_new_state = {}
for old_k, new_k in zip(list(_adap_state.keys()), list(adaptive_surrogate.state_dict().keys())):
    _adap_new_state[new_k] = _adap_state[old_k]
adaptive_surrogate.load_state_dict(_adap_new_state)
adaptive_surrogate.eval()
print('Adaptive surrogate loaded successfully.')

def pgd_adaptive(model, X, y_np, epsilon=0.1, alpha=None, n_iter=10):
    model.eval()
    if alpha is None: alpha = epsilon / 4
    y_t    = torch.from_numpy(y_np).long().to(DEVICE)
    X_orig = torch.from_numpy(X).float().to(DEVICE)
    X_adv  = X_orig.clone().detach()
    for _ in range(n_iter):
        X_adv.requires_grad_(True)
        loss = F.cross_entropy(model(X_adv), y_t)
        grad = torch.autograd.grad(loss, X_adv)[0]
        X_adv = X_adv.detach() + alpha * grad.sign()
        X_adv = (X_orig + torch.clamp(X_adv-X_orig,-epsilon,epsilon)).detach()
    return X_adv.cpu().numpy()

X_sample_adap = pgd_adaptive(adaptive_surrogate, X_sample_clean, y_sample)
print(f'Adaptive adversarial sample: {X_sample_adap.shape}')

# Build SHAP explainers
print('\nBuilding SHAP explainers (TreeExplainer -- exact, fast)...')
# TreeExplainer needs a raw Booster/sklearn model it recognizes internally --
# BoosterWrapper is a custom class it won't recognize, so pass the underlying
# boosters loaded in the Setup cell instead.
explainer_std = shap.TreeExplainer(_booster_std)
explainer_def = shap.TreeExplainer(_booster_def)

# Compute SHAP values for all conditions
print('Computing SHAP values:')
print('  1/4 Standard IDS, clean...')
shap_std_clean = explainer_std.shap_values(X_sample_clean)
print('  2/4 Standard IDS, PGD...')
shap_std_pgd   = explainer_std.shap_values(X_sample_pgd)
print('  3/4 Robust IDS, clean...')
shap_def_clean = explainer_def.shap_values(X_sample_clean)
print('  4/4 Robust IDS, adaptive PGD...')
shap_def_adap  = explainer_def.shap_values(X_sample_adap)
print('All SHAP values computed.')


## 3. ESI and PCR computation functions

In [ ]:
def mean_abs_shap(shap_values):
    if isinstance(shap_values, list):
        stacked = np.stack(shap_values, axis=2)
    else:
        stacked = shap_values
    return np.abs(stacked).mean(axis=2)


def compute_esi(shap_clean, shap_adv, X_clean, X_adv):
    phi_clean = mean_abs_shap(shap_clean)
    phi_adv   = mean_abs_shap(shap_adv)
    delta_phi = np.linalg.norm(phi_adv - phi_clean, axis=1)
    delta_x   = np.linalg.norm(X_adv - X_clean, axis=1)
    valid = delta_x > 1e-8
    esi = np.ones(len(delta_x))
    esi[valid] = 1.0 - delta_phi[valid] / (delta_x[valid] + 1e-8)
    return np.clip(esi, 0, 1)


def compute_pcr(shap_values, physics_idx):
    phi = mean_abs_shap(shap_values)
    physics_attr = phi[:, physics_idx].sum(axis=1)
    total_attr   = phi.sum(axis=1)
    return np.where(total_attr > 1e-8, physics_attr / total_attr, 0.0)


print('Computing ESI...')
esi_std_pgd  = compute_esi(shap_std_clean, shap_std_pgd,
                             X_sample_clean, X_sample_pgd)
esi_def_adap = compute_esi(shap_def_clean, shap_def_adap,
                             X_sample_clean, X_sample_adap)

print('Computing PCR...')
pcr_std_clean = compute_pcr(shap_std_clean, PHYSICS_IDX)
pcr_std_pgd   = compute_pcr(shap_std_pgd,   PHYSICS_IDX)
pcr_def_clean = compute_pcr(shap_def_clean, PHYSICS_IDX)
pcr_def_adap  = compute_pcr(shap_def_adap,  PHYSICS_IDX)

print('ESI/PCR computed.')


## 4. ESI results

In [ ]:
attack_sample = (y_sample != 0)

print('=' * 65)
print('ESI (Explainability Stability Index) Results')
print('ESI=1: perfect stability | ESI=0: complete instability')
print('=' * 65)
print()

print(f"{'Condition':<42} {'Mean ESI':>9} {'Std':>7} {'Min':>7}")
print('-' * 65)
print(f"{'Standard IDS + PGD (attack classes)':<42} "
      f"{esi_std_pgd[attack_sample].mean():>9.4f} "
      f"{esi_std_pgd[attack_sample].std():>7.4f} "
      f"{esi_std_pgd[attack_sample].min():>7.4f}")
print(f"{'Robust IDS + Adaptive PGD (attack)':<42} "
      f"{esi_def_adap[attack_sample].mean():>9.4f} "
      f"{esi_def_adap[attack_sample].std():>7.4f} "
      f"{esi_def_adap[attack_sample].min():>7.4f}")

print()
print('Per-class ESI (Standard IDS + PGD):')
print(f"  {'Class':<14} {'Mean ESI':>10} {'Std':>8}")
for cls_idx, cls_name in enumerate(class_names):
    mask = (y_sample == cls_idx)
    if mask.sum() == 0: continue
    print(f"  {cls_name:<14} {esi_std_pgd[mask].mean():>10.4f} {esi_std_pgd[mask].std():>8.4f}")

print()
print('Per-class ESI (Robust IDS + Adaptive PGD):')
print(f"  {'Class':<14} {'Mean ESI':>10} {'Std':>8}")
for cls_idx, cls_name in enumerate(class_names):
    mask = (y_sample == cls_idx)
    if mask.sum() == 0: continue
    print(f"  {cls_name:<14} {esi_def_adap[mask].mean():>10.4f} {esi_def_adap[mask].std():>8.4f}")


## 5. PCR results

In [ ]:
print('=' * 65)
print('PCR (Physics Constrained Ratio) Results')
print(f'Physics features: {PHYSICS_FEATURES}')
print('=' * 65)
print()

conditions = [
    ('Standard IDS, clean',       pcr_std_clean),
    ('Standard IDS, PGD attack',  pcr_std_pgd),
    ('Robust IDS, clean',         pcr_def_clean),
    ('Robust IDS, Adaptive PGD',  pcr_def_adap),
]

print(f"{'Condition':<35} {'Mean PCR':>10} {'Std':>8} {'Change':>10}")
print('-' * 65)
baseline_pcr = pcr_std_clean.mean()
for name, pcr in conditions:
    change = pcr.mean() - baseline_pcr
    print(f"{name:<35} {pcr.mean():>10.4f} {pcr.std():>8.4f} {change:>+10.4f}")

print()
print('Per-class PCR (Standard IDS, clean vs. PGD):')
print(f"  {'Class':<14} {'Clean PCR':>10} {'PGD PCR':>10} {'Change':>10}")
for cls_idx, cls_name in enumerate(class_names):
    mask = (y_sample == cls_idx)
    if mask.sum() == 0: continue
    c_pcr = pcr_std_clean[mask].mean()
    a_pcr = pcr_std_pgd[mask].mean()
    print(f"  {cls_name:<14} {c_pcr:>10.4f} {a_pcr:>10.4f} {a_pcr-c_pcr:>+10.4f}")

pcr_change = pcr_std_pgd.mean() - pcr_std_clean.mean()
print(f'\nOverall PCR change under PGD: {pcr_change:+.4f}')
if pcr_change > 0:
    print('-> PCR INCREASES under attack: model relies MORE on physics features')
    print('   under adversarial conditions. Physics-grounded features are harder')
    print('   to perturb than cyber features, so their relative attribution rises.')
else:
    print('-> PCR DECREASES under attack: adversarial perturbation shifts')
    print('   attribution toward cyber features.')


## 6. Top feature attribution analysis

In [ ]:
phi_std_clean = mean_abs_shap(shap_std_clean)
phi_std_pgd   = mean_abs_shap(shap_std_pgd)
phi_def_clean = mean_abs_shap(shap_def_clean)

mean_attr_clean = phi_std_clean.mean(axis=0)
top10_idx = np.argsort(mean_attr_clean)[::-1][:10]

print('Top 10 features by mean |SHAP| attribution (Standard IDS, clean):')
print(f"  {'Feature':<28} {'Clean':>8} {'PGD':>8} {'Change':>8} {'Type':>10}")
print('  ' + '-' * 65)
for idx in top10_idx:
    feat = FEATURE_COLS[idx]
    c_attr = phi_std_clean[:, idx].mean()
    p_attr = phi_std_pgd[:, idx].mean()
    ftype = 'physics' if feat in PHYSICS_FEATURES else             ('cyber' if feat.startswith('cyber_') else 'physical')
    print(f"  {feat:<28} {c_attr:>8.4f} {p_attr:>8.4f} {p_attr-c_attr:>+8.4f} {ftype:>10}")


## 7. Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: ESI by class (Standard IDS + PGD)
class_esi_pgd = [esi_std_pgd[y_sample==i] for i in range(N_CLASSES) if (y_sample==i).sum()>0]
axes[0,0].boxplot(class_esi_pgd, labels=class_names, patch_artist=True)
axes[0,0].set_title('ESI Distribution by Class (Standard IDS + PGD)')
axes[0,0].set_ylabel('ESI Score')
axes[0,0].set_ylim([0, 1.05])
axes[0,0].tick_params(axis='x', rotation=30)
axes[0,0].grid(alpha=0.3)
axes[0,0].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='ESI=0.5')
axes[0,0].legend(fontsize=8)

# Plot 2: PCR across conditions
pcr_means  = [pcr_std_clean.mean(), pcr_std_pgd.mean(),
              pcr_def_clean.mean(), pcr_def_adap.mean()]
pcr_stds   = [pcr_std_clean.std(), pcr_std_pgd.std(),
              pcr_def_clean.std(), pcr_def_adap.std()]
pcr_labels = ['Std Clean', 'Std PGD', 'Robust Clean', 'Robust Adap']
colors = ['steelblue', 'red', 'green', 'orange']
axes[0,1].bar(pcr_labels, pcr_means, yerr=pcr_stds, color=colors, alpha=0.7, capsize=5)
axes[0,1].set_title('PCR (Physics Constrained Ratio) Across Conditions')
axes[0,1].set_ylabel('PCR')
axes[0,1].set_ylim([0, max(pcr_means)*1.3])
axes[0,1].grid(alpha=0.3)
axes[0,1].tick_params(axis='x', rotation=20)

# Plot 3: Top feature attribution clean vs PGD
top8_idx   = np.argsort(phi_std_clean.mean(axis=0))[::-1][:8]
feat_names = [FEATURE_COLS[i][:18] for i in top8_idx]
c_attrs    = phi_std_clean[:, top8_idx].mean(axis=0)
p_attrs    = phi_std_pgd[:, top8_idx].mean(axis=0)
x_pos = np.arange(len(top8_idx))
axes[1,0].bar(x_pos-0.2, c_attrs, 0.4, label='Clean', color='steelblue', alpha=0.7)
axes[1,0].bar(x_pos+0.2, p_attrs, 0.4, label='PGD',   color='red',       alpha=0.7)
axes[1,0].set_xticks(x_pos)
axes[1,0].set_xticklabels(feat_names, rotation=40, ha='right')
axes[1,0].set_title('Mean |SHAP| Attribution: Clean vs. PGD (Top 8 Features)')
axes[1,0].set_ylabel('Mean |SHAP|')
axes[1,0].legend()
axes[1,0].grid(alpha=0.3)

# Plot 4: ESI comparison Standard vs Robust per class
esi_std_cls  = [esi_std_pgd[y_sample==i].mean()  for i in range(N_CLASSES)]
esi_def_cls  = [esi_def_adap[y_sample==i].mean() for i in range(N_CLASSES)]
x_pos2 = np.arange(N_CLASSES)
axes[1,1].bar(x_pos2-0.2, esi_std_cls, 0.4, label='Std+PGD',      color='red',   alpha=0.7)
axes[1,1].bar(x_pos2+0.2, esi_def_cls, 0.4, label='Robust+Adap',  color='green', alpha=0.7)
axes[1,1].set_xticks(x_pos2)
axes[1,1].set_xticklabels(class_names, rotation=30)
axes[1,1].set_title('ESI per Class: Standard+PGD vs. Robust+Adaptive')
axes[1,1].set_ylabel('Mean ESI')
axes[1,1].set_ylim([0, 1.05])
axes[1,1].legend()
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'esi_pcr_analysis_baseline{SUFFIX}.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: esi_pcr_analysis_baseline{SUFFIX}.png')


## 8. Save all ESI/PCR results to Drive

In [ ]:
attack_sample = (y_sample != 0)

results = {
    'esi_std_pgd_mean':     float(esi_std_pgd[attack_sample].mean()),
    'esi_std_pgd_std':      float(esi_std_pgd[attack_sample].std()),
    'esi_def_adap_mean':    float(esi_def_adap[attack_sample].mean()),
    'esi_def_adap_std':     float(esi_def_adap[attack_sample].std()),
    'pcr_std_clean_mean':   float(pcr_std_clean.mean()),
    'pcr_std_pgd_mean':     float(pcr_std_pgd.mean()),
    'pcr_def_clean_mean':   float(pcr_def_clean.mean()),
    'pcr_def_adap_mean':    float(pcr_def_adap.mean()),
    'pcr_change_pgd':       float(pcr_std_pgd.mean() - pcr_std_clean.mean()),
    'per_class_esi_std_pgd':{
        class_names[i]: float(esi_std_pgd[y_sample==i].mean())
        for i in range(N_CLASSES) if (y_sample==i).sum()>0
    },
    'per_class_esi_def_adap':{
        class_names[i]: float(esi_def_adap[y_sample==i].mean())
        for i in range(N_CLASSES) if (y_sample==i).sum()>0
    },
    'per_class_pcr_clean':{
        class_names[i]: float(pcr_std_clean[y_sample==i].mean())
        for i in range(N_CLASSES) if (y_sample==i).sum()>0
    },
    'per_class_pcr_pgd':{
        class_names[i]: float(pcr_std_pgd[y_sample==i].mean())
        for i in range(N_CLASSES) if (y_sample==i).sum()>0
    },
    'top_features_clean':{
        FEATURE_COLS[i]: float(phi_std_clean[:,i].mean())
        for i in np.argsort(phi_std_clean.mean(axis=0))[::-1][:10]
    },
    'physics_idx': PHYSICS_IDX,
    'physics_features': PHYSICS_FEATURES,
    'n_shap_samples': int(len(sample_idx)),
}

joblib.dump(results, f'esi_pcr_results_baseline{SUFFIX}.pkl')

print(f'NB8-baseline artifacts saved locally (track={"BINARY" if BINARY else "MULTICLASS"}):')
for f in [f'esi_pcr_results_baseline{SUFFIX}.pkl', f'esi_pcr_analysis_baseline{SUFFIX}.png']:
    if os.path.exists(f):
        size = os.path.getsize(f)/1e6
        print(f'  ok  {f}  ({size:.1f} MB)')
    else:
        print(f'  MISSING  {f}')

print()
print('ESI/PCR Summary:')
print(f"  PCR (clean):           {results['pcr_std_clean_mean']:.4f}")
print(f"  PCR (PGD):             {results['pcr_std_pgd_mean']:.4f}")
print(f"  PCR change:            {results['pcr_change_pgd']:+.4f}")
print(f"  ESI (Std + PGD):       {results['esi_std_pgd_mean']:.4f}")
print(f"  ESI (Robust + Adap):   {results['esi_def_adap_mean']:.4f}")
